In [18]:
import pandas as pd
import numpy as np

print("Libraries imported successfully!")

Libraries imported successfully!


In [19]:
import pandas as pd

# Load the retail orders dataset
orders = pd.read_csv("../Data/retail-orders-raw.csv")

# Load the data dictionary
dictionary = pd.read_csv("../Data/retail-data-dictionary.csv")

print("Retail orders loaded successfully!")
print("Rows:", orders.shape[0])
print("Columns:", orders.shape[1])

display(orders)

Retail orders loaded successfully!
Rows: 12
Columns: 9


,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status
0,RT-1001,2026-01-03,Student,Chennai,Learning Kit,2,799,10.0,Paid
1,RT-1002,03/01/2026,Fresher,Bengaluru,Course Access,1,1499,0.0,paid
2,RT-1003,2026-01-05,student,Chennai,Course Access,1,1499,NaN,Pending
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid
4,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid
5,RT-1005,2026-01-09,Fresher,NaN,Mentor Session,1,999,0.0,Failed
6,RT-1006,2026-13-10,Student,Pune,Learning Kit,-1,799,10.0,Paid
7,RT-1007,2026-01-12,Professional,Mumbai,Mentor Session,2,999,105.0,Paid
8,RT-1008,2026-01-14,Fresher,Bengaluru,Course Access,two,1499,0.0,Pending
9,RT-1009,2026-01-16,Student,Chennai,Learning Kit,1,799,0.0,Paid


In [20]:
print("Dataset Shape:")
print(orders.shape)

print("\nColumn Names:")
print(orders.columns.tolist())

print("\nData Types:")
print(orders.dtypes)

print("\nFirst 5 Rows:")
display(orders.head())

Dataset Shape:
(12, 9)

Column Names:
['order_id', 'order_date', 'customer_segment', 'city', 'category', 'quantity', 'unit_price', 'discount_pct', 'payment_status']

Data Types:
order_id                str
order_date              str
customer_segment        str
city                    str
category                str
quantity                str
unit_price            int64
discount_pct        float64
payment_status          str
dtype: object

First 5 Rows:


,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status
0,RT-1001,2026-01-03,Student,Chennai,Learning Kit,2,799,10.0,Paid
1,RT-1002,03/01/2026,Fresher,Bengaluru,Course Access,1,1499,0.0,paid
2,RT-1003,2026-01-05,student,Chennai,Course Access,1,1499,NaN,Pending
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid
4,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid


In [21]:
missing_count = orders.isnull().sum()

missing_report = pd.DataFrame({
    "Column": orders.columns,
    "Missing Count": missing_count.values,
    "Missing Percentage": (
        missing_count.values / len(orders) * 100
    ).round(2)
})

display(missing_report)

,Column,Missing Count,Missing Percentage
0,order_id,0,0.00
1,order_date,1,8.33
2,customer_segment,0,0.00
3,city,1,8.33
4,category,0,0.00
5,quantity,0,0.00
6,unit_price,0,0.00
7,discount_pct,1,8.33
8,payment_status,0,0.00


In [22]:
# Check duplicate order IDs

duplicate_count = orders["order_id"].duplicated().sum()

print("Duplicate Order IDs:", duplicate_count)

if duplicate_count == 0:
    print("Uniqueness Check: PASS")
else:
    print("Uniqueness Check: FAIL")

Duplicate Order IDs: 1
Uniqueness Check: FAIL


In [23]:
# Show duplicate order records

duplicates = orders[
    orders["order_id"].duplicated(keep=False)
]

display(duplicates)

,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid
4,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid


In [24]:

orders["parsed_order_date"] = pd.to_datetime(
    orders["order_date"],
    errors="coerce",
    dayfirst=False
)

# Check invalid or missing dates
invalid_dates = orders["parsed_order_date"].isna()

print("Invalid or Missing Dates:", invalid_dates.sum())

# Show problematic rows
display(
    orders.loc[
        invalid_dates,
        ["order_id", "order_date"]
    ]
)

if invalid_dates.sum() == 0:
    print("Date Validity Check: PASS")
else:
    print("Date Validity Check: FAIL")

Invalid or Missing Dates: 3


,order_id,order_date
1,RT-1002,03/01/2026
6,RT-1006,2026-13-10
11,RT-1011,NaN


Date Validity Check: FAIL


In [25]:
orders["quantity_num"] = pd.to_numeric(
    orders["quantity"],
    errors="coerce"
)

orders["unit_price_num"] = pd.to_numeric(
    orders["unit_price"],
    errors="coerce"
)

# Quantity check
invalid_quantity = (
    orders["quantity_num"].isna() |
    (orders["quantity_num"] <= 0)
)

# Unit Price check
invalid_price = (
    orders["unit_price_num"].isna() |
    (orders["unit_price_num"] <= 0)
)

print("Invalid Quantity:", invalid_quantity.sum())
print("Invalid Unit Price:", invalid_price.sum())

if invalid_quantity.sum() == 0:
    print("Quantity Validity Check: PASS")
else:
    print("Quantity Validity Check: FAIL")

if invalid_price.sum() == 0:
    print("Unit Price Validity Check: PASS")
else:
    print("Unit Price Validity Check: FAIL")

Invalid Quantity: 2
Invalid Unit Price: 0
Quantity Validity Check: FAIL
Unit Price Validity Check: PASS


In [26]:
orders["discount_num"] = pd.to_numeric(
    orders["discount_pct"],
    errors="coerce"
)
invalid_discount = (
    orders["discount_num"].isna() |
    (orders["discount_num"] < 0) |
    (orders["discount_num"] > 100)
)

# Payment Status allowed values
allowed_status = ["Paid", "Pending"]

invalid_payment = (
    orders["payment_status"].isna() |
    ~orders["payment_status"].isin(allowed_status)
)

print("Invalid or Missing Discount:", invalid_discount.sum())
print("Invalid or Missing Payment Status:", invalid_payment.sum())

if invalid_discount.sum() == 0:
    print("Discount Validity Check: PASS")
else:
    print("Discount Validity Check: FAIL")

if invalid_payment.sum() == 0:
    print("Payment Status Validity Check: PASS")
else:
    print("Payment Status Validity Check: FAIL")

Invalid or Missing Discount: 2
Invalid or Missing Payment Status: 3
Discount Validity Check: FAIL
Payment Status Validity Check: FAIL


In [27]:
# Allowed customer segments
allowed_segments = [
    "Student",
    "Fresher",
    "Professional"
]
orders["segment_normalized"] = (
    orders["customer_segment"]
    .astype("string")
    .str.strip()
    .str.title()
)

# Invalid segment check
invalid_segment = (
    orders["segment_normalized"].isna() |
    ~orders["segment_normalized"].isin(allowed_segments)
)

print("Invalid Customer Segment:", invalid_segment.sum())

display(
    orders.loc[
        invalid_segment,
        ["order_id", "customer_segment"]
    ]
)

if invalid_segment.sum() == 0:
    print("Customer Segment Check: PASS")
else:
    print("Customer Segment Check: FAIL")

Invalid Customer Segment: 0


,order_id,customer_segment


Customer Segment Check: PASS


In [28]:

missing_city = orders["city"].isna() | (
    orders["city"].astype("string").str.strip() == ""
)

# Payment Status allowed values
allowed_payment = ["Paid", "Pending"]

invalid_payment_status = (
    orders["payment_status"].isna() |
    ~orders["payment_status"].isin(allowed_payment)
)

print("Missing City:", missing_city.sum())
print("Invalid Payment Status:", invalid_payment_status.sum())

if missing_city.sum() == 0:
    print("City Completeness Check: PASS")
else:
    print("City Completeness Check: FAIL")

if invalid_payment_status.sum() == 0:
    print("Payment Status Check: PASS")
else:
    print("Payment Status Check: FAIL")

Missing City: 1
Invalid Payment Status: 3
City Completeness Check: FAIL
Payment Status Check: FAIL


In [29]:

quality_summary = pd.DataFrame({
    "Data Quality Check": [
        "Completeness",
        "Order ID Uniqueness",
        "Order Date Validity",
        "Quantity Validity",
        "Unit Price Validity",
        "Discount Validity",
        "Payment Status Validity",
        "Customer Segment Consistency",
        "City Completeness"
    ],
    
    "Result": [
        "FAIL",
        "FAIL",
        "FAIL",
        "FAIL",
        "PASS",
        "FAIL",
        "PASS",
        "PASS",
        "FAIL"
    ],
    
    "Status": [
        "Needs Attention",
        "Critical",
        "Critical",
        "Critical",
        "Pass",
        "High Priority",
        "Pass",
        "Pass",
        "Needs Attention"
    ]
})

display(quality_summary)

print("\n" + "=" * 60)
print("OVERALL DATA QUALITY SUMMARY")
print("=" * 60)

total_checks = len(quality_summary)
passed_checks = (quality_summary["Result"] == "PASS").sum()
failed_checks = (quality_summary["Result"] == "FAIL").sum()

print("Total Checks :", total_checks)
print("Passed Checks:", passed_checks)
print("Failed Checks:", failed_checks)

if failed_checks == 0:
    print("\nOverall Data Quality: PASS")
else:
    print("\nOverall Data Quality: FAIL")
    print("Action Required: Data quality issues must be reviewed.")


,Data Quality Check,Result,Status
0,Completeness,FAIL,Needs Attention
1,Order ID Uniqueness,FAIL,Critical
2,Order Date Validity,FAIL,Critical
3,Quantity Validity,FAIL,Critical
4,Unit Price Validity,PASS,Pass
5,Discount Validity,FAIL,High Priority
6,Payment Status Validity,PASS,Pass
7,Customer Segment Consistency,PASS,Pass
8,City Completeness,FAIL,Needs Attention



OVERALL DATA QUALITY SUMMARY
Total Checks : 9
Passed Checks: 3
Failed Checks: 6

Overall Data Quality: FAIL
Action Required: Data quality issues must be reviewed.


In [30]:

total_rows = len(orders)

# Completeness
required_columns = [
    "order_id",
    "order_date",
    "customer_segment",
    "city",
    "category",
    "quantity",
    "unit_price",
    "discount_pct",
    "payment_status"
]

total_missing = orders[required_columns].isnull().sum().sum()
total_cells = total_rows * len(required_columns)
completeness_rate = (1 - total_missing / total_cells) * 100

# Uniqueness
duplicate_rate = (
    orders["order_id"].duplicated().sum() / total_rows
) * 100

# Strict ISO Date Validity
date_text = orders["order_date"].astype("string").str.strip()

valid_iso_format = (
    date_text.notna()
    & date_text.str.match(r"^\d{4}-\d{2}-\d{2}$", na=False)
)

parsed_dates = pd.to_datetime(
    date_text,
    format="%Y-%m-%d",
    errors="coerce"
)

valid_dates = valid_iso_format & parsed_dates.notna()

invalid_date_count = (~valid_dates).sum()

# Quantity
invalid_quantity_count = invalid_quantity.sum()

# Unit Price
invalid_price_count = invalid_price.sum()

# Discount
invalid_discount_count = invalid_discount.sum()

# Customer Segment
invalid_segment_count = invalid_segment.sum()

# Payment Status
invalid_payment_count = invalid_payment_status.sum()

# City
missing_city_count = missing_city.sum()


# Create Summary
quality_summary = pd.DataFrame({
    "Quality Dimension": [
        "Completeness",
        "Order ID Uniqueness",
        "Order Date Validity",
        "Quantity Validity",
        "Unit Price Validity",
        "Discount Validity",
        "Customer Segment Consistency",
        "Payment Status Consistency",
        "City Completeness"
    ],
    
    "Result": [
        f"{completeness_rate:.2f}%",
        f"{duplicate_rate:.2f}%",
        f"{invalid_date_count} invalid",
        f"{invalid_quantity_count} invalid",
        f"{invalid_price_count} invalid",
        f"{invalid_discount_count} invalid",
        f"{invalid_segment_count} invalid",
        f"{invalid_payment_count} invalid",
        f"{missing_city_count} missing"
    ],
    
    "Status": [
        "PASS" if completeness_rate >= 98 else "FAIL",
        "PASS" if duplicate_rate == 0 else "FAIL",
        "PASS" if invalid_date_count == 0 else "FAIL",
        "PASS" if invalid_quantity_count == 0 else "FAIL",
        "PASS" if invalid_price_count == 0 else "FAIL",
        "PASS" if invalid_discount_count == 0 else "FAIL",
        "PASS" if invalid_segment_count / total_rows <= 0.01 else "FAIL",
        "PASS" if invalid_payment_count / total_rows <= 0.01 else "FAIL",
        "PASS" if missing_city_count == 0 else "FAIL"
    ]
})

print("==============================================")
print("OVERALL DATA QUALITY SUMMARY")
print("==============================================")

display(quality_summary)

critical_checks = [
    invalid_date_count == 0,
    invalid_quantity_count == 0,
    invalid_price_count == 0,
    duplicate_rate == 0
]

if all(critical_checks):
    print("\nOverall Critical Status: PASS")
else:
    print("\nOverall Critical Status: FAIL")

print("==============================================")

OVERALL DATA QUALITY SUMMARY


,Quality Dimension,Result,Status
0,Completeness,97.22%,FAIL
1,Order ID Uniqueness,8.33%,FAIL
2,Order Date Validity,3 invalid,FAIL
3,Quantity Validity,2 invalid,FAIL
4,Unit Price Validity,0 invalid,PASS
5,Discount Validity,2 invalid,FAIL
6,Customer Segment Consistency,0 invalid,PASS
7,Payment Status Consistency,3 invalid,FAIL
8,City Completeness,1 missing,FAIL



Overall Critical Status: FAIL


In [31]:


import os

os.makedirs("../Output", exist_ok=True)

output_file = "../Output/data_quality_summary.csv"

quality_summary.to_csv(
    output_file,
    index=False
)

print("Data Quality Summary saved successfully!")
print("File:", output_file)

display(quality_summary)

Data Quality Summary saved successfully!
File: ../Output/data_quality_summary.csv


,Quality Dimension,Result,Status
0,Completeness,97.22%,FAIL
1,Order ID Uniqueness,8.33%,FAIL
2,Order Date Validity,3 invalid,FAIL
3,Quantity Validity,2 invalid,FAIL
4,Unit Price Validity,0 invalid,PASS
5,Discount Validity,2 invalid,FAIL
6,Customer Segment Consistency,0 invalid,PASS
7,Payment Status Consistency,3 invalid,FAIL
8,City Completeness,1 missing,FAIL


In [32]:
import os
from datetime import datetime

file_path = "../Data/retail-orders-raw.csv"

modified_time = os.path.getmtime(file_path)

last_modified = datetime.fromtimestamp(modified_time)
current_time = datetime.now()

age_hours = (
    current_time - last_modified
).total_seconds() / 3600

print("Last Modified:", last_modified)
print("Data Age:", round(age_hours, 2), "hours")

if age_hours <= 24:
    freshness_status = "PASS"
else:
    freshness_status = "FAIL"

print("Freshness Check:", freshness_status)

Last Modified: 2026-09-12 18:52:02.112572
Data Age: 6.55 hours
Freshness Check: PASS


In [33]:

critical_failed = (
    duplicate_rate > 0
    or invalid_date_count > 0
    or invalid_quantity_count > 0
    or invalid_price_count > 0
)

high_severity_failed = (
    completeness_rate < 98
    or invalid_discount_count / total_rows > 0
    or invalid_segment_count / total_rows > 0.01
    or invalid_payment_count / total_rows > 0.01
    or missing_city_count / total_rows > 0.02
)

if critical_failed:
    final_status = "FAIL"
elif high_severity_failed:
    final_status = "WARNING"
else:
    final_status = "PASS"

print("==============================================")
print("FINAL DATA QUALITY STATUS")
print("==============================================")
print("Status:", final_status)

if final_status == "FAIL":
    print("\nAction: Dataset is NOT TRUSTED for KPI reporting.")
elif final_status == "WARNING":
    print("\nAction: Dataset can be used with documented limitations.")
else:
    print("\nAction: Dataset is trusted for KPI reporting.")

print("==============================================")

FINAL DATA QUALITY STATUS
Status: FAIL

Action: Dataset is NOT TRUSTED for KPI reporting.


In [34]:


print("=" * 55)
print("KPI DICTIONARY & DATA QUALITY CONTRACT")
print("FINAL EXECUTIVE SUMMARY")
print("=" * 55)

print("\nDataset Information")
print("-------------------")
print("Total Rows:", total_rows)
print("Total Columns:", len(required_columns))

print("\nData Quality Results")
print("--------------------")
print(f"Completeness: {completeness_rate:.2f}%")
print(f"Duplicate Order IDs: {orders['order_id'].duplicated().sum()}")
print(f"Invalid Dates: {invalid_date_count}")
print(f"Invalid Quantity: {invalid_quantity_count}")
print(f"Invalid Unit Price: {invalid_price_count}")
print(f"Invalid/Missing Discount: {invalid_discount_count}")
print(f"Invalid Customer Segment: {invalid_segment_count}")
print(f"Invalid Payment Status: {invalid_payment_count}")
print(f"Missing City: {missing_city_count}")
print(f"Freshness Status: {freshness_status}")

print("\nFinal Dataset Status")
print("--------------------")
print("STATUS:", final_status)

if final_status == "FAIL":
    print("The dataset is NOT TRUSTED for KPI reporting.")
    print("Critical data-quality issues must be corrected first.")
elif final_status == "WARNING":
    print("The dataset can be used with documented limitations.")
else:
    print("The dataset is TRUSTED for KPI reporting.")

print("\nRecommended Action")
print("-------------------")

if final_status == "FAIL":
    print("1. Stop KPI publication.")
    print("2. Notify the Operations Manager.")
    print("3. Correct duplicate/invalid records.")
    print("4. Rerun all data-quality checks.")
    print("5. Publish KPIs only after critical checks PASS.")
else:
    print("Continue regular KPI monitoring and daily refresh.")

print("=" * 55)

KPI DICTIONARY & DATA QUALITY CONTRACT
FINAL EXECUTIVE SUMMARY

Dataset Information
-------------------
Total Rows: 12
Total Columns: 9

Data Quality Results
--------------------
Completeness: 97.22%
Duplicate Order IDs: 1
Invalid Dates: 3
Invalid Quantity: 2
Invalid Unit Price: 0
Invalid/Missing Discount: 2
Invalid Customer Segment: 0
Invalid Payment Status: 3
Missing City: 1
Freshness Status: PASS

Final Dataset Status
--------------------
STATUS: FAIL
The dataset is NOT TRUSTED for KPI reporting.
Critical data-quality issues must be corrected first.

Recommended Action
-------------------
1. Stop KPI publication.
2. Notify the Operations Manager.
3. Correct duplicate/invalid records.
4. Rerun all data-quality checks.
5. Publish KPIs only after critical checks PASS.
